In [21]:
#RAG : Indexing , Retrieving , Augumentation , Generation 

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_openai.embeddings import OpenAIEmbeddings

In [12]:
#Step 1a : Indexing( Document loader)
video_id='ZDa-Z5JzLYM'
try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript = " ".join(snippet.text for snippet in fetched_transcript)
    print(transcript)
except TranscriptsDisabled:
    print("Transcript is disabled for this video.")

Hey, everybody. How's [it] going in this series of videos? We'll be learning how to create and use classes within python and how object-oriented concepts are applied within the language now There's a lot to cover when working with classes, so I'm going to break these up into several different videos We'll cover the basics of creating and instantiating classes will learn about inheritance class and instance variables static methods and class methods and Several other topics So breaking these up in several videos will allow us to focus on specific topics in each video so in this video We'll be learning the basics of creating and instantiating simple classes, but first Why should we even use classes now this isn't just specific to Python you can see classes being used throughout most modern programming languages, and there's a good reason for that they allow us to Logically group our data and functions in a way that's easy to [reuse] and also easy to build upon if need be Now just a quick

In [ ]:
#Step 1b : Indexing( Text splitting )
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript]) # this will create a document object
len(chunks)
chunks

[Document(metadata={}, page_content="Hey, everybody. How's [it] going in this series of videos? We'll be learning how to create and use classes within python and how object-oriented concepts are applied within the language now There's a lot to cover when working with classes, so I'm going to break these up into several different videos We'll cover the basics of creating and instantiating classes will learn about inheritance class and instance variables static methods and class methods and Several other topics So breaking these up in several videos will allow us to focus on specific topics in each video so in this video We'll be learning the basics of creating and instantiating simple classes, but first Why should we even use classes now this isn't just specific to Python you can see classes being used throughout most modern programming languages, and there's a good reason for that they allow us to Logically group our data and functions in a way that's easy to [reuse] and also easy to b

In [14]:
#Step 1c and 1d : Indexing(Embedding Generation and Storing in vector store)
embeddings=OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore=FAISS.from_documents(chunks,embeddings)

# to print the embedding value for each chunk 
vectorstore.index_to_docstore_id

{0: 'b2f82fcf-3082-4779-8e5e-b2feda6f32d4',
 1: '2b4dea05-f9f3-44a0-ae1a-49ba41d7f664',
 2: '41b943ec-9f00-4bc5-a2b0-03546cd086a8',
 3: 'fbe7de94-d69e-4b2d-a037-4014932305a6',
 4: '213381b7-68bc-40d1-a441-f278bf8b4172',
 5: 'fb011360-8abc-4af3-8ac9-8e2dfe9084b5',
 6: 'ac981a18-795d-421f-b38d-7a928e47cf1e',
 7: '3ab8fdd2-89d1-43a0-bae5-3325e4952d38',
 8: 'e3dfef8e-7d9b-4e6f-9472-26ed774704df',
 9: '674f569e-e4d9-4753-8371-494bbf2b6b1f',
 10: 'cc318844-814a-46e6-bf10-37430348a132',
 11: '5931bda4-41cc-4082-9853-1a7ab7b680b4',
 12: 'c6b15c71-c073-47b0-8fc8-39e3e4d36334',
 13: '740d7a59-1248-48a7-8794-2872058f23fe',
 14: 'e655438d-1942-4b51-ba1e-d0101a8f07bb',
 15: 'e1ac7cda-5911-4d6e-9a45-65adf93baf52',
 16: '7532e7bc-01b3-45e0-ac1a-f0d2040e3818',
 17: '2c4e63fd-cecd-4ed7-be6b-21b5ff77d015'}

In [ ]:
vectorstore.get_by_ids(['69b8aa71-3e85-43c4-b4a1-9d34e76a02ab'])

In [16]:
#Step2- Retrieve

retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':4})
retriever.invoke("what is class and objects")


[Document(id='b2f82fcf-3082-4779-8e5e-b2feda6f32d4', metadata={}, page_content="Hey, everybody. How's [it] going in this series of videos? We'll be learning how to create and use classes within python and how object-oriented concepts are applied within the language now There's a lot to cover when working with classes, so I'm going to break these up into several different videos We'll cover the basics of creating and instantiating classes will learn about inheritance class and instance variables static methods and class methods and Several other topics So breaking these up in several videos will allow us to focus on specific topics in each video so in this video We'll be learning the basics of creating and instantiating simple classes, but first Why should we even use classes now this isn't just specific to Python you can see classes being used throughout most modern programming languages, and there's a good reason for that they allow us to Logically group our data and functions in a wa

In [17]:
#Step3 - Augumentation( combining most relavant chunk with prompt )
llm=ChatOpenAI(model='gpt-3.5-turbo',temperature=0.2)
prompt=PromptTemplate(
    template="""you are a helpfull assistant,
    answer only from the provided transcript context.
    If the context is insufficient , just say you don't know.
    {context}
    question:{question}""",
    input_variables=['context','question']
)

question="what is class and objects"
retrieved_docs=retriever.invoke(question)
context_text="\n\n".join(doc.page_content for doc in retrieved_docs) # to concatinate the pagecontent from all 4 retrieved docs
final_prompt=prompt.invoke({'context':retrieved_docs,'question':question})
print(final_prompt)



text='you are a helpfull assistant,\n    answer only from the provided transcript context.\n    If the context is insufficient , just say you don\'t know.\n    [Document(id=\'b2f82fcf-3082-4779-8e5e-b2feda6f32d4\', metadata={}, page_content="Hey, everybody. How\'s [it] going in this series of videos? We\'ll be learning how to create and use classes within python and how object-oriented concepts are applied within the language now There\'s a lot to cover when working with classes, so I\'m going to break these up into several different videos We\'ll cover the basics of creating and instantiating classes will learn about inheritance class and instance variables static methods and class methods and Several other topics So breaking these up in several videos will allow us to focus on specific topics in each video so in this video We\'ll be learning the basics of creating and instantiating simple classes, but first Why should we even use classes now this isn\'t just specific to Python you ca

In [18]:
#Step4- Generation

answer=llm.invoke(final_prompt)
print(answer)

content='In the provided transcript context, a class is described as a blueprint for creating instances, and each unique employee created using the employee class is an instance of that class. Each instance has unique data and methods associated with it.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 988, 'total_tokens': 1032, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EBefH0zCr1wPK5NQAB64zWRIlytMY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019ff084-545b-79f0-8fec-d2c5f188778d-0' usage_metadata={'input_tokens': 988, 'output_tokens': 44, 'total_tokens': 1032, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outp